# 🔍 Text Classifier — Exploratory Data Analysis
Explore the dataset, visualize class distribution, and inspect cleaned text.

In [ ]:
import sys
sys.path.append('..')  # allow imports from project root

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from src.preprocess import clean_text, preprocess_dataframe

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
df = pd.read_csv('../data/raw/news.csv')
print(f'Shape: {df.shape}')
df.head()

## 📊 Class Distribution

In [ ]:
label_counts = df['label'].value_counts()
ax = label_counts.plot(kind='bar', color=sns.color_palette('Set2'))
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 🧹 Inspect Cleaned Text

In [ ]:
df_clean = preprocess_dataframe(df)
df_clean[['text', 'clean_text', 'label']].head(10)

## 📏 Text Length Distribution

In [ ]:
df_clean['word_count'] = df_clean['clean_text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_clean['word_count'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Word Count Distribution (Cleaned)')
axes[0].set_xlabel('Words')

df_clean.boxplot(column='word_count', by='label', ax=axes[1])
axes[1].set_title('Word Count by Class')
axes[1].set_xlabel('Category')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 🔤 Top Words Per Class

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, label in enumerate(df_clean['label'].unique()):
    texts = df_clean[df_clean['label'] == label]['clean_text'].tolist()
    vec = CountVectorizer(max_features=15, min_df=1)
    X = vec.fit_transform(texts)
    word_freq = dict(zip(vec.get_feature_names_out(), X.toarray().sum(axis=0)))
    top = sorted(word_freq.items(), key=lambda x: -x[1])[:15]
    words, freqs = zip(*top)
    axes[i].barh(words[::-1], freqs[::-1], color=sns.color_palette('Set2')[i])
    axes[i].set_title(f'Top words — {label}', fontweight='bold')

plt.tight_layout()
plt.show()